In [1]:
!pip install -q transformers torch sentencepiece pyspellchecker gradio pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 46.4 MB/s eta 0:00:00


In [2]:
import re
import difflib
import string
import pandas as pd
import numpy as np

import torch
import gradio as gr

from spellchecker import SpellChecker
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [4]:
spell = SpellChecker()

print("Spelling correction system loaded.")

Spelling correction system loaded.


In [5]:
word = "beutiful"

corrected_word = spell.correction(word)

print("Original :", word)
print("Corrected:", corrected_word)

Original : beutiful
Corrected: beautiful


In [6]:
MODEL_NAME = "vennify/t5-base-grammar-correction"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model = model.to(device)

model.eval()

print("AI Grammar Correction Model Loaded.")

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

AI Grammar Correction Model Loaded.


In [7]:
def grammar_correction(text):
    if not text or not text.strip():
        return ""

    input_text = "grammar: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=512,
            num_beams=5,
            early_stopping=True
        )

    corrected_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return corrected_text

In [8]:
text = "I am goin to scholl yesterday."

corrected = grammar_correction(text)

print("Original :", text)
print("Corrected:", corrected)

Original : I am goin to scholl yesterday.
Corrected: I am going to school yesterday.


In [9]:
def spelling_correction(text):
    if not text or not text.strip():
        return ""

    words = re.findall(r'\b[a-zA-Z]+\b', text)

    corrected_text = text

    for word in words:
        if len(word) <= 2:
            continue

        # Only check alphabetic words
        if word.isalpha():
            corrected_word = spell.correction(word.lower())

            if corrected_word and corrected_word != word.lower():

                # Preserve capitalization
                if word[0].isupper():
                    corrected_word = corrected_word.capitalize()

                corrected_text = re.sub(
                    rf'\b{re.escape(word)}\b',
                    corrected_word,
                    corrected_text,
                    count=1
                )

    return corrected_text

In [10]:
text = "I hav a beutiful hous."

corrected = spelling_correction(text)

print("Original :", text)
print("Corrected:", corrected)

Original : I hav a beutiful hous.
Corrected: I have a beautiful house.


In [11]:
def autocorrect_basic(text):

    if not text or not text.strip():
        return ""

    # Step 1: spelling correction
    text = spelling_correction(text)

    # Step 2: grammar/context correction
    text = grammar_correction(text)

    return text

In [12]:
text = "i hav a beutiful hous"

result = autocorrect_basic(text)

print("Original :", text)
print("Corrected:", result)

Original : i hav a beutiful hous
Corrected: I have a beautiful house.


In [13]:
test_sentences = [
    "I hav a beutiful hous.",
    "She dont like this movie.",
    "i am going to college tomorow."
]

for sentence in test_sentences:
    print("\nOriginal :", sentence)
    print("Corrected:", autocorrect_basic(sentence))


Original : I hav a beutiful hous.
Corrected: I have a beautiful house.

Original : She dont like this movie.
Corrected: She doesn't like this movie.

Original : i am going to college tomorow.
Corrected: I am going to college tomorrow.


In [14]:
def punctuation_capitalization_correction(text):

    if not text or not text.strip():
        return ""

    # Remove unnecessary spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Capitalize the first character
    if text:
        text = text[0].upper() + text[1:]

    # Capitalize the first letter after . ! ?
    text = re.sub(
        r'([.!?]\s+)([a-z])',
        lambda m: m.group(1) + m.group(2).upper(),
        text
    )

    # Add punctuation at the end if missing
    if text and text[-1] not in ".!?":
        text += "."

    return text

In [15]:
test_text = "i am going to college tomorrow"

result = punctuation_capitalization_correction(test_text)

print("Original :", test_text)
print("Corrected:", result)

Original : i am going to college tomorrow
Corrected: I am going to college tomorrow.


In [16]:
test_sentences = [
    "i am a student",
    "she is going to school",
    "what are you doing",
    "this is amazing"
]

for sentence in test_sentences:
    print("\nOriginal :", sentence)
    print("Corrected:", punctuation_capitalization_correction(sentence))


Original : i am a student
Corrected: I am a student.

Original : she is going to school
Corrected: She is going to school.

Original : what are you doing
Corrected: What are you doing.

Original : this is amazing
Corrected: This is amazing.


In [17]:
def autocorrect_basic(text):

    if not text or not text.strip():
        return ""

    # Step 1: Spelling correction
    text = spelling_correction(text)

    # Step 2: Grammar correction
    text = grammar_correction(text)

    # Step 3: Punctuation and capitalization
    text = punctuation_capitalization_correction(text)

    return text

In [18]:
test_sentences = [
    "I hav a beutiful hous",
    "She dont like this movie",
    "i am going to college tomorow",
    "he is a good student",
    "what are you doing"
]

for sentence in test_sentences:
    print("\nOriginal :", sentence)
    print("Corrected:", autocorrect_basic(sentence))


Original : I hav a beutiful hous
Corrected: I have a beautiful house.

Original : She dont like this movie
Corrected: She doesn't like this movie.

Original : i am going to college tomorow
Corrected: I am going to college tomorrow.

Original : he is a good student
Corrected: He is a good student.

Original : what are you doing
Corrected: What are you doing?


In [19]:
def get_corrections(original, corrected):

    original_words = original.split()
    corrected_words = corrected.split()

    matcher = difflib.SequenceMatcher(
        None,
        original_words,
        corrected_words
    )

    corrections = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():

        if tag == "replace":
            corrections.append({
                "Original": " ".join(original_words[i1:i2]),
                "Corrected": " ".join(corrected_words[j1:j2]),
                "Type": "Replacement"
            })

        elif tag == "delete":
            corrections.append({
                "Original": " ".join(original_words[i1:i2]),
                "Corrected": "",
                "Type": "Deletion"
            })

        elif tag == "insert":
            corrections.append({
                "Original": "",
                "Corrected": " ".join(corrected_words[j1:j2]),
                "Type": "Insertion"
            })

    return corrections

In [20]:
original = "I hav a beutiful hous"
corrected = autocorrect_basic(original)

corrections = get_corrections(original, corrected)

print("Original :", original)
print("Corrected:", corrected)

print("\nCorrections:")

for correction in corrections:
    print(
        f"{correction['Original']} -> "
        f"{correction['Corrected']} "
        f"({correction['Type']})"
    )

Original : I hav a beutiful hous
Corrected: I have a beautiful house.

Corrections:
hav -> have (Replacement)
beutiful hous -> beautiful house. (Replacement)


In [21]:
def correction_statistics(original, corrected):

    corrections = get_corrections(original, corrected)

    total_corrections = len(corrections)

    return {
        "Original Words": len(original.split()),
        "Corrected Words": len(corrected.split()),
        "Total Corrections": total_corrections
    }

In [22]:
stats = correction_statistics(original, corrected)

print(stats)

{'Original Words': 5, 'Corrected Words': 5, 'Total Corrections': 2}


In [23]:
def classify_correction(original_word, corrected_word):

    original_clean = original_word.strip(string.punctuation)
    corrected_clean = corrected_word.strip(string.punctuation)

    # Capitalization error
    if (
        original_clean.lower() == corrected_clean.lower()
        and original_clean != corrected_clean
    ):
        return "Capitalization"

    # Punctuation error
    if original_clean == corrected_clean and original_word != corrected_word:
        return "Punctuation"

    # Spelling error
    if len(original_clean) >= 3 and len(corrected_clean) >= 3:
        return "Spelling"

    return "Grammar"

In [24]:
def get_detailed_corrections(original, corrected):

    original_words = original.split()
    corrected_words = corrected.split()

    matcher = difflib.SequenceMatcher(
        None,
        original_words,
        corrected_words
    )

    corrections = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():

        if tag == "replace":

            old_text = " ".join(original_words[i1:i2])
            new_text = " ".join(corrected_words[j1:j2])

            correction_type = classify_correction(
                old_text,
                new_text
            )

            corrections.append({
                "Original": old_text,
                "Corrected": new_text,
                "Type": correction_type
            })

        elif tag == "delete":

            corrections.append({
                "Original": " ".join(original_words[i1:i2]),
                "Corrected": "",
                "Type": "Deletion"
            })

        elif tag == "insert":

            corrections.append({
                "Original": "",
                "Corrected": " ".join(corrected_words[j1:j2]),
                "Type": "Insertion"
            })

    return corrections

In [25]:
original = "I hav a beutiful hous"
corrected = autocorrect_basic(original)

corrections = get_detailed_corrections(
    original,
    corrected
)

print("Original :", original)
print("Corrected:", corrected)

print("\nDetailed Corrections:")

for correction in corrections:
    print(
        f"{correction['Original']} -> "
        f"{correction['Corrected']} "
        f"[{correction['Type']}]"
    )

Original : I hav a beutiful hous
Corrected: I have a beautiful house.

Detailed Corrections:
hav -> have [Spelling]
beutiful hous -> beautiful house. [Spelling]


In [26]:
def generate_correction_report(original, corrected):

    corrections = get_detailed_corrections(
        original,
        corrected
    )

    spelling_count = 0
    grammar_count = 0
    punctuation_count = 0
    capitalization_count = 0

    for correction in corrections:

        correction_type = correction["Type"]

        if correction_type == "Spelling":
            spelling_count += 1

        elif correction_type == "Grammar":
            grammar_count += 1

        elif correction_type == "Punctuation":
            punctuation_count += 1

        elif correction_type == "Capitalization":
            capitalization_count += 1

    report = {
        "Original Text": original,
        "Corrected Text": corrected,
        "Total Corrections": len(corrections),
        "Spelling Errors": spelling_count,
        "Grammar Errors": grammar_count,
        "Punctuation Errors": punctuation_count,
        "Capitalization Errors": capitalization_count
    }

    return report

In [27]:
original = "I hav a beutiful hous"
corrected = autocorrect_basic(original)

report = generate_correction_report(
    original,
    corrected
)

for key, value in report.items():
    print(f"{key}: {value}")

Original Text: I hav a beutiful hous
Corrected Text: I have a beautiful house.
Total Corrections: 2
Spelling Errors: 2
Grammar Errors: 0
Punctuation Errors: 0
Capitalization Errors: 0


In [28]:
def autocorrect_system(text):

    if not text or not text.strip():
        return {
            "Original Text": "",
            "Corrected Text": "",
            "Total Corrections": 0,
            "Spelling Errors": 0,
            "Grammar Errors": 0,
            "Punctuation Errors": 0,
            "Capitalization Errors": 0
        }

    # Generate corrected text
    corrected = autocorrect_basic(text)

    # Generate report
    report = generate_correction_report(
        text,
        corrected
    )

    return report

In [29]:
result = autocorrect_system(
    "She dont like this movie"
)

for key, value in result.items():
    print(f"{key}: {value}")

Original Text: She dont like this movie
Corrected Text: She doesn't like this movie.
Total Corrections: 2
Spelling Errors: 1
Grammar Errors: 0
Punctuation Errors: 1
Capitalization Errors: 0


In [30]:
test_sentences = [
    "I hav a beutiful hous",
    "She dont like this movie",
    "i am going to college tomorow",
    "he is a good student",
    "what are you doing",
    "This sentence is already correct."
]

for sentence in test_sentences:

    result = autocorrect_system(sentence)

    print("\n" + "=" * 60)
    print("Original :", result["Original Text"])
    print("Corrected:", result["Corrected Text"])
    print("Corrections:", result["Total Corrections"])


Original : I hav a beutiful hous
Corrected: I have a beautiful house.
Corrections: 2

Original : She dont like this movie
Corrected: She doesn't like this movie.
Corrections: 2

Original : i am going to college tomorow
Corrected: I am going to college tomorrow.
Corrections: 2

Original : he is a good student
Corrected: He is a good student.
Corrections: 2

Original : what are you doing
Corrected: What are you doing?
Corrections: 2

Original : This sentence is already correct.
Corrected: This sentence is already correct.
Corrections: 0


In [31]:
def is_already_correct(original, corrected):

    original_normalized = re.sub(r'\s+', ' ', original.strip())
    corrected_normalized = re.sub(r'\s+', ' ', corrected.strip())

    return original_normalized == corrected_normalized

In [32]:
text = "This sentence is already correct."

corrected = autocorrect_basic(text)

print("Original :", text)
print("Corrected:", corrected)
print("Already Correct:", is_already_correct(text, corrected))

Original : This sentence is already correct.
Corrected: This sentence is already correct.
Already Correct: True


In [33]:
def calculate_correction_score(original, corrected):

    original_words = original.split()
    corrected_words = corrected.split()

    if len(original_words) == 0:
        return 100.0

    matcher = difflib.SequenceMatcher(
        None,
        original_words,
        corrected_words
    )

    similarity = matcher.ratio()

    score = similarity * 100

    return round(score, 2)

In [34]:
original = "I hav a beutiful hous"
corrected = autocorrect_basic(original)

score = calculate_correction_score(
    original,
    corrected
)

print("Original :", original)
print("Corrected:", corrected)
print("Correction Score:", score, "%")

Original : I hav a beutiful hous
Corrected: I have a beautiful house.
Correction Score: 40.0 %


In [35]:
def autocorrect_system(text):

    if not text or not text.strip():
        return {
            "Original Text": "",
            "Corrected Text": "",
            "Status": "No input provided",
            "Correction Score": 0.0,
            "Total Corrections": 0,
            "Spelling Errors": 0,
            "Grammar Errors": 0,
            "Punctuation Errors": 0,
            "Capitalization Errors": 0
        }

    corrected = autocorrect_basic(text)

    report = generate_correction_report(
        text,
        corrected
    )

    score = calculate_correction_score(
        text,
        corrected
    )

    if is_already_correct(text, corrected):
        status = "No correction needed"
    else:
        status = "Corrections applied"

    report["Status"] = status
    report["Correction Score"] = score

    return report

In [36]:
result = autocorrect_system(
    "I hav a beutiful hous"
)

for key, value in result.items():
    print(f"{key}: {value}")

Original Text: I hav a beutiful hous
Corrected Text: I have a beautiful house.
Total Corrections: 2
Spelling Errors: 2
Grammar Errors: 0
Punctuation Errors: 0
Capitalization Errors: 0
Status: Corrections applied
Correction Score: 40.0


In [37]:
result = autocorrect_system(
    "This sentence is already correct."
)

for key, value in result.items():
    print(f"{key}: {value}")

Original Text: This sentence is already correct.
Corrected Text: This sentence is already correct.
Total Corrections: 0
Spelling Errors: 0
Grammar Errors: 0
Punctuation Errors: 0
Capitalization Errors: 0
Status: No correction needed
Correction Score: 100.0


In [38]:
def autocorrect_interface(text):

    result = autocorrect_system(text)

    original = result["Original Text"]
    corrected = result["Corrected Text"]

    status = result["Status"]
    score = result["Correction Score"]

    total = result["Total Corrections"]
    spelling = result["Spelling Errors"]
    grammar = result["Grammar Errors"]
    punctuation = result["Punctuation Errors"]
    capitalization = result["Capitalization Errors"]

    statistics = f"""
### Correction Statistics

**Status:** {status}

**Correction Score:** {score}%

**Total Corrections:** {total}

**Spelling Errors:** {spelling}

**Grammar Errors:** {grammar}

**Punctuation Errors:** {punctuation}

**Capitalization Errors:** {capitalization}
"""

    return corrected, statistics

In [39]:
with gr.Blocks(title="AI-Powered Autocorrect Tool") as demo:

    gr.Markdown(
        """
        # ✨ AI-Powered Autocorrect Tool

        Correct spelling, grammar, capitalization, and punctuation
        using AI-powered text correction.
        """
    )

    with gr.Row():

        input_text = gr.Textbox(
            label="Enter Text",
            placeholder="Type a sentence with spelling or grammar mistakes...",
            lines=8
        )

        output_text = gr.Textbox(
            label="Corrected Text",
            lines=8
        )

    correct_button = gr.Button(
        "✨ Correct Text"
    )

    statistics = gr.Markdown(
        "Correction statistics will appear here."
    )

    correct_button.click(
        fn=autocorrect_interface,
        inputs=input_text,
        outputs=[output_text, statistics]
    )

    gr.Examples(
        examples=[
            ["I hav a beutiful hous"],
            ["She dont like this movie"],
            ["i am going to college tomorow"],
            ["what are you doing"],
            ["This sentence is already correct."]
        ],
        inputs=input_text
    )

In [46]:
demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9962906ae0602c35bf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
